# 👟 ARShoe M1v2 13-Keypoint High-Speed Pipeline in Google Colab

This notebook trains and evaluates **ARShoe M1v2 (13-KP)** using the two consolidated, high-performance pipeline files:
1. **`shoes_vto/train_13kp_pipeline.py`**: High-speed training with GPU grid caching, batch_size=64, AMP FP16, and live every-10-epoch visualizations.
2. **`shoes_vto/eval_13kp_pipeline.py`**: Full validation benchmark with OpenPose PAF-led assembly, PCK@0.20/0.10/0.05 metrics, foot recall, and side-by-side rendering.

### Step 1: Verify GPU Environment & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q pyyaml tqdm albumentations opencv-python matplotlib scipy torchvision

### Step 2: Set Working Directory & Pull Latest Code

In [ ]:
import os
if os.path.exists('/content/shoes_vto_v2/shoes_vto'):
    %cd /content/shoes_vto_v2/shoes_vto
elif os.path.exists('/content/shoes_vto'):
    %cd /content/shoes_vto
print('Working Directory:', os.getcwd())
!git pull origin deletekp6and12

### Step 3: Convert `shuffled_v4_1` (16-KP) to 13-KP Format
- Strips `ball_top` (5), `ankle_center` (12), and `shin_mid` (15)
- Uses instant symlinks for images to avoid duplicating storage
- Output path: `/content/shoes_vto_v2/shoes_vto/data/shuffled_v4_1_13kp`

In [ ]:
!python shoes_vto/tools/convert_dataset_to_13kp.py \
    --src /content/shoes_vto_v2/shoes_vto/data/shuffled_v4_1 \
    --dst /content/shoes_vto_v2/shoes_vto/data/shuffled_v4_1_13kp \
    --symlink

### Step 4: Launch High-Speed 100-Epoch Training (~10s per epoch on T4)
- **Grid caching**: Reuses coordinate grids on GPU, removing allocator stalls.
- **Batch Size 64**: Maximizes CUDA core utilization (~3 GB VRAM).
- **AMP FP16**: 2x speedup via Tensor Cores.
- **Live Visualizations**: Saved to `epoch_visualizations/epoch_XXX_vis.jpg` every 10 epochs.

In [ ]:
!python shoes_vto/train_13kp_pipeline.py \
    --dataset_root /content/shoes_vto_v2/shoes_vto/data/shuffled_v4_1_13kp \
    --output_dir shoes_vto/outputs/m1v2_13kp_stage2_paf_fix \
    --epochs 100 \
    --batch_size 64 \
    --lr 0.001 \
    --num_workers 2 \
    --vis_interval 10

### Step 5: Display Live Training Visualizations

In [ ]:
from IPython.display import Image, display
import glob

vis_files = sorted(glob.glob('shoes_vto/outputs/m1v2_13kp_stage2_paf_fix/epoch_visualizations/epoch_*_vis.jpg'))
if vis_files:
    print(f'Latest Visualization: {vis_files[-1]}')
    display(Image(vis_files[-1]))
else:
    print('No visualization images found yet.')

### Step 6: Full Validation Benchmark with OpenPose PAF-Led Assembly
Evaluates the new `best.pth` checkpoint on all validation images with PCK@0.20, PCK@0.10, PCK@0.05, and foot recall.

In [ ]:
!python shoes_vto/eval_13kp_pipeline.py \
    --checkpoint shoes_vto/outputs/m1v2_13kp_stage2_paf_fix/checkpoints/best.pth \
    --valid_dir /content/shoes_vto_v2/shoes_vto/data/shuffled_v4_1_13kp/valid \
    --output_dir shoes_vto/outputs/validation_eval_shuffled_v4_1

### Step 7: Package Checkpoints & Visualizations for Download

In [ ]:
!zip -r m1v2_13kp_stage2_paf_fix_results.zip \
    shoes_vto/outputs/m1v2_13kp_stage2_paf_fix/checkpoints/best.pth \
    shoes_vto/outputs/m1v2_13kp_stage2_paf_fix/epoch_visualizations/ \
    shoes_vto/outputs/validation_eval_shuffled_v4_1/
print('✅ Results packaged to m1v2_13kp_stage2_paf_fix_results.zip!')